In [13]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [15]:
load_dotenv()

True

In [16]:
model = ChatGoogleGenerativeAI(model='gemini-3.5-flash-lite')

In [17]:
class BatsmanState(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int

    sr: float
    bpb: float
    boundary_percent: float
    summary: str

In [48]:
def calculate_sr(state: BatsmanState) -> BatsmanState:
    sr = (state['runs']/state['balls']) * 100

    # state['sr'] = round(sr, 2)

    return {'sr': round(sr, 2)}

In [49]:
def calculate_bpb(state: BatsmanState) -> BatsmanState:
    bpb = state['balls']/(state['fours'] + state['sixes'])

    # state['bpb'] = round(bpb, 2)
    
    return {'bpb': round(bpb, 2)}

In [50]:
def calculate_boundary_percent(state: BatsmanState) -> BatsmanState:
    boundary_percent = (((state['fours'] * 4) + (state['sixes'] * 6))/state['runs']) * 100

    # state['boundary_percent'] = round(boundary_percent, 2)

    return {'boundary_percent': round(boundary_percent, 2)}

In [54]:
def summary(state: BatsmanState) -> BatsmanState:
    summary = f"""
Strike Rate - {state['sr']}\n
Balls per boundary - {state['bpb']}\n
Boundary percent - {state['boundary_percent']}
"""
    # state['summary'] = summary
    
    return {'summary': summary}

In [56]:
graph = StateGraph(BatsmanState)

graph.add_node('calculate_sr', calculate_sr)
graph.add_node('calculate_bpb', calculate_bpb)
graph.add_node('calculate_boundary_percent', calculate_boundary_percent)
graph.add_node('summary', summary)

graph.add_edge(START, 'calculate_sr')
graph.add_edge(START, 'calculate_boundary_percent')
graph.add_edge(START, 'calculate_bpb')
graph.add_edge('calculate_sr', 'summary')
graph.add_edge('calculate_boundary_percent', 'summary')
graph.add_edge('calculate_bpb', 'summary')
graph.add_edge('summary', END)

workflow = graph.compile()

In [57]:
initial_state = {
    'runs' : 100,
    'balls' : 30,
    'fours' : 6,
    'sixes' : 10
}

final_state = workflow.invoke(initial_state)

print(final_state)

{'runs': 100, 'balls': 30, 'fours': 6, 'sixes': 10, 'sr': 333.33, 'bpb': 1.88, 'boundary_percent': 84.0, 'summary': '\nStrike Rate - 333.33\n\nBalls per boundary - 1.88\n\nBoundary percent - 84.0\n'}
